# Sequence Engineering & LSTM Input Design
## Financial Fraud Detection System

This notebook handles the transformation of preprocessed transaction data into sequences suitable for LSTM models.

### Key Design Decision
Based on data analysis: **Fraudsters have exactly 1 transaction each** (no history). Therefore, we use **GLOBAL time-based sequences** ordered by `step` across all transactions, rather than per-user sequences.

### Workflow
1. Load cleaned preprocessed dataset
2. Create global time-ordered sequences
3. Prepare LSTM input format (samples, timesteps, features)
4. Handle class imbalance in sequences
5. Generate batch-ready data for training

### Setup and Import Libraries

In [1]:
print("=" * 60)
print("SEQUENCE ENGINEERING FOR LSTM")
print("=" * 60)

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

# Data handling
import joblib
import json
from pathlib import Path

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("All libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

SEQUENCE ENGINEERING FOR LSTM
All libraries imported successfully!
TensorFlow version: 2.21.0
NumPy version: 2.4.4
Pandas version: 3.0.2


### Load and Explore the Clean Dataset

In [2]:
print("=" * 60)
print("LOAD AND EXPLORE CLEAN DATASET")
print("=" * 60)

# Setup paths
PROJECT_ROOT = Path.cwd().parent
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
PROCESSED_DATA_DIR = ARTIFACTS_DIR / 'processed'

# Load the cleaned dataset
data_path = PROCESSED_DATA_DIR / 'cleaned_data.parquet'

if not data_path.exists():
    print(f"ERROR: Dataset not found at {data_path}")
    print("Please run data_preprocessing.ipynb first")
else:
    df = pd.read_parquet(data_path)
    print(f"Dataset loaded successfully from {data_path.name}")

print("\n" + "=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)

print(f"\nDataset shape: {df.shape}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

print("\nFirst 5 rows:")
display(df.head())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
missing = df.isnull().sum()
if missing.sum() == 0:
    print("Dataset has no missing values")
else:
    print(missing[missing > 0])

print("\nDataset splits:")
print(df['split'].value_counts())

print("\nFraud distribution:")
fraud_dist = df.groupby('split')['isFraud'].value_counts().unstack(fill_value=0)
fraud_dist['fraud_rate_%'] = (fraud_dist[1] / (fraud_dist[0] + fraud_dist[1]) * 100).round(4)
print(fraud_dist)

print("\nFeature information:")
feature_list_path = PROCESSED_DATA_DIR / 'feature_names.txt'
if feature_list_path.exists():
    with open(feature_list_path, 'r') as f:
        feature_cols = [line.strip() for line in f.readlines()]
    print(f"Number of features: {len(feature_cols)}")
    print(f"Features: {feature_cols[:10]}...")  # Show first 10
else:
    print("Feature names file not found")
    feature_cols = [col for col in df.columns if col not in ['nameOrig', 'nameDest', 'isFraud', 'isFlaggedFraud', 'split']]
    print(f"Inferred features: {len(feature_cols)}")

print("\nDataset exploration complete")

LOAD AND EXPLORE CLEAN DATASET
Dataset loaded successfully from cleaned_data.parquet

DATASET OVERVIEW

Dataset shape: (6362620, 23)
Rows: 6,362,620
Columns: 23

First 5 rows:


,nameOrig,nameDest,isFraud,isFlaggedFraud,step,amount,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER,hour_of_cycle,day_of_cycle,sender_transaction_count,is_first_transaction,dest_fraud_rate,type_CASH_IN_avg_deviation,type_CASH_OUT_avg_deviation,type_DEBIT_avg_deviation,type_PAYMENT_avg_deviation,type_TRANSFER_avg_deviation,high_amount_to_fraud_dest,split
0,C1000000639,C785826240,0,0,0.039773,0.108090,0,1,0,0,0,-1.462597,0.084296,-0.032017,1,-0.050916,0.000000,68270.145658,0.0,0.000000,-0.0,-0.058428,train
1,C1000001337,M216466820,0,0,-0.185020,-0.294330,0,0,0,1,0,-3.313667,-0.084537,-0.032017,1,-0.050916,-0.000000,-0.000000,-0.0,-9890.668399,-0.0,-0.058428,train
2,C1000002591,C503690069,0,0,-0.086673,0.137091,1,0,0,0,0,-0.074295,-0.084537,-0.032017,1,-0.050916,93047.944766,0.000000,0.0,0.000000,-0.0,-0.058428,train
3,C1000003372,C1840417793,0,0,-0.536260,-0.265383,1,0,0,0,0,1.776775,-0.591037,-0.032017,1,-0.050916,-148300.595234,-0.000000,0.0,0.000000,-0.0,-0.058428,train
4,C1000004053,C1128041097,0,0,0.587707,0.052564,0,1,0,0,0,-0.074295,0.590796,-0.032017,1,-0.050916,0.000000,34973.325658,0.0,0.000000,-0.0,-0.058428,train



Data types:
nameOrig                           str
nameDest                           str
isFraud                          int64
isFlaggedFraud                   int64
step                           float64
amount                         float64
type_CASH_IN                     int64
type_CASH_OUT                    int64
type_DEBIT                       int64
type_PAYMENT                     int64
type_TRANSFER                    int64
hour_of_cycle                  float64
day_of_cycle                   float64
sender_transaction_count       float64
is_first_transaction             int64
dest_fraud_rate                float64
type_CASH_IN_avg_deviation     float64
type_CASH_OUT_avg_deviation    float64
type_DEBIT_avg_deviation       float64
type_PAYMENT_avg_deviation     float64
type_TRANSFER_avg_deviation    float64
high_amount_to_fraud_dest      float64
split                              str
dtype: object

Missing values:
Dataset has no missing values

Dataset splits:
split
train 

### Prepare Data for Global Time-Based Sequences

**Design Rationale:**
- Fraudsters have exactly 1 transaction (no per-user history)
- Use GLOBAL temporal ordering (by `step`)
- Create sequences of N consecutive transactions
- Each sequence is labeled by the fraud status of the CENTER/TARGET transaction

In [3]:
print("=" * 60)
print("PREPARE DATA FOR SEQUENCES")
print("=" * 60)

# Extract feature columns (exclude identifiers and labels)
exclude_cols = {'nameOrig', 'nameDest', 'isFraud', 'isFlaggedFraud', 'split'}
feature_cols = [col for col in df.columns if col not in exclude_cols]

print(f"\nFeatures for sequences: {len(feature_cols)}")
print(f"Features: {feature_cols}")

# Sort by time globally (by step)
df_sorted = df.sort_values('step').reset_index(drop=True)

print("\nDataset sorted by 'step' (global time)")
print(f"Step range: {df_sorted['step'].min():.0f} to {df_sorted['step'].max():.0f}")

# Extract features and labels
X = df_sorted[feature_cols].values  # Shape: (n_samples, n_features)
y = df_sorted['isFraud'].values     # Shape: (n_samples,)
splits = df_sorted['split'].values   # Track train/val/test

print(f"\nFeatures shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Splits: {np.unique(splits, return_counts=True)}")

print("\nData prepared for sequence creation")

PREPARE DATA FOR SEQUENCES

Features for sequences: 18
Features: ['step', 'amount', 'type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER', 'hour_of_cycle', 'day_of_cycle', 'sender_transaction_count', 'is_first_transaction', 'dest_fraud_rate', 'type_CASH_IN_avg_deviation', 'type_CASH_OUT_avg_deviation', 'type_DEBIT_avg_deviation', 'type_PAYMENT_avg_deviation', 'type_TRANSFER_avg_deviation', 'high_amount_to_fraud_dest']

Dataset sorted by 'step' (global time)
Step range: -2 to 4

Features shape: (6362620, 18)
Labels shape: (6362620,)
Splits: (array(['test', 'train', 'val'], dtype=object), array([ 954393, 4453837,  954390]))

Data prepared for sequence creation


### Create Sequences Using Sliding Window

### Prepare LSTM Input Format and Normalize Features

### Visualize Sample Sequences

### Create Batch Data Generators for Training